chunking参考文档：https://community.databricks.com/t5/technical-blog/the-ultimate-guide-to-chunking-strategies-for-rag-applications/ba-p/113089

# RAG 分块（Chunking）策略总结（Databricks Guide）

本文系统性地介绍了 RAG（Retrieval-Augmented Generation）中的分块策略，强调：
Chunking 不是简单的数据预处理，而是直接决定 RAG 效果的核心设计环节。

---

### 1. 为什么 Chunking 很重要
- 将长文本拆分为可嵌入（embedding）、检索（retrieval）的最小单元。
- 直接影响：
  - 检索准确率（Recall / Precision）
  - LLM 生成质量
  - 计算成本与响应速度
- 用于解决 LLM context window 限制问题。

---

### 2. 核心分块策略 + 适用场景

#### 固定长度分块（Fixed-size Chunking）
**定义：**
- 按固定 token / 字符长度切分（如 500 tokens）。

**优点：**
- 实现简单、速度快。
- 易于规模化处理。

**缺点：**
- 容易破坏语义（可能截断句子/段落）。

**适用场景：**
- 非结构化数据（日志、邮件、聊天记录）。
- 数据清洗初期（baseline RAG）。
- 文档没有明显结构的情况。

#### 语义分块（Semantic Chunking）
**定义：**
- 按语义边界切分（句子 / 段落 / 主题）。

**优点：**
- 保持语义完整性。
- 提升检索精度。

**缺点：**
- 实现复杂（需要 NLP 或 embedding 辅助）。

**适用场景：**
- FAQ / 知识库构建。
- 技术文档（如 biotech 产品说明）。
- 邮件中的 Q&A 提取。

#### 递归分块（Recursive Chunking）
**定义：**
- 分层切分：文档 -> 章节 -> 段落 -> 句子。

**优点：**
- 兼顾结构和长度控制。
- 更稳定、鲁棒性强。

**缺点：**
- 工程复杂度略高。

**适用场景：**
- 长文档（PDF、报告）。
- 有标题结构的文档。
- 生产级 RAG 系统。

#### 自适应分块（Adaptive / Dynamic Chunking）
**定义：**
- 根据内容动态调整 chunk 大小。

**优点：**
- 灵活性高。
- 可同时优化 recall 和 precision。

**缺点：**
- 调参复杂、实现难度高。

**适用场景：**
- 多数据源系统（邮件 + PDF + 知识库）。
- 高性能 RAG 系统（需要优化 latency + cost）。
- 企业级应用。

---

### 3. 关键设计要点
- **Chunk size**
  - 太小 -> 上下文丢失。
  - 太大 -> 噪声增加、检索不精准。
- **Overlap（重叠）**
  - 保证上下文连续性（推荐 10%–20%）。
- **语义完整性**
  - 每个 chunk 应该表达一个完整信息单元。
- **Metadata（非常重要）**
  - 如：sender / date / product / intent。
  - 能显著提升检索效果。

---

### 4. 核心结论
> RAG 的效果在很大程度上取决于 chunking 设计。
> 好的 chunking = 更精准的检索 + 更高质量的回答。

---

### 5. 结合当前 Email Agent 项目的建议

| 阶段 | 推荐策略 |
| :--- | :--- |
| 原始邮件 | Fixed-size |
| 清洗后邮件 | Semantic |
| 知识库构建 | Semantic + overlap |
| 上线系统 | Recursive / Hybrid |

**最优实践：**
语义分块 + overlap + metadata = 高质量 RAG 系统